# scTASL tutorial: scRNA-seq and scATAC-seq integration

This notebook trains the scTASL integration model, generates cell and feature
embeddings, visualizes the joint cell space, and evaluates integration quality.

**Expected input files** (under `dataset/<DATASET>/`):

- `rna_processed.h5ad`: processed RNA counts with a `counts` layer
- `atac_processed.h5ad`: processed ATAC counts with a `counts` layer
- `graph_data.pkl`: the precomputed gene–peak prior graph

This tutorial dataset includes `obs["cell_type"]` so that the notebook can
show labeled UMAPs and supervised evaluation metrics. In real applications,
cell-type labels are not required for training the integration model; skip the
label-based evaluation cells if they are unavailable. For paired integration,
RNA and ATAC cells must have identical observation names in identical order.

> Run this notebook from the repository root so that the relative paths and
> local Python imports resolve correctly.


## 1. Setup

Import the required packages and project utilities. Explicit imports make the
notebook easier to audit and prevent accidental name collisions.


In [ ]:
import os
from pathlib import Path

import anndata
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import torch
from IPython.display import display
import warnings
warnings.filterwarnings("ignore")

from model.integration_model import Integration_Model
from utils.common.data_configure import configure_dataset, load_omics_inputs
from utils.common.metrics import (
    adjusted_rand_index,
    avg_silhouette_width_batch,
    avg_silhouette_width_label,
    compute_omics_entropy,
    compute_bcs,
    foscttm,
    graph_connectivity,
    kBET,
    norm_lisi_omics,
    norm_lisi_label,
    normalized_mutual_information,
    transfer_accuracy,
)
from utils.common.plots import (
    cell_type_palette,
    knn_transfer_confusion_rna_to_atac_percent,
)

# Keep text editable when PDF figures are opened in vector-graphics software.
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
sc.set_figure_params(dpi=80, facecolor="white")


### Configure the run

Change `DATASET` to integrate another preprocessed dataset. `IS_PAIRED=True`
enables paired-cell training and paired-cell evaluation. The random seed makes
NumPy and PyTorch operations reproducible where deterministic implementations
are available.


In [ ]:
DATASET = "PBMC-10k"
TASK = "integration"
IS_PAIRED = True
RANDOM_SEED = 1

DATA_DIR = Path("dataset") / DATASET
RESULT_DIR = Path("results") / DATASET / TASK
ADATA_DIR = RESULT_DIR / "adata"
FIGURE_DIR = RESULT_DIR / "figure"
METRICS_DIR = RESULT_DIR / "metrics"

for output_dir in (RESULT_DIR, ADATA_DIR, FIGURE_DIR, METRICS_DIR):
    output_dir.mkdir(parents=True, exist_ok=True)

np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

print(f"Dataset: {DATASET}")
print(f"Results: {RESULT_DIR.resolve()}")
print(f"Compute device: {'CUDA' if torch.cuda.is_available() else 'CPU'}")


## 2. Load and validate the inputs

The validation below fails early with an actionable message when a required
file, count layer, annotation, or paired-cell ordering is missing. Only load
`graph_data.pkl` when it comes from this project or another trusted source,
because Python pickle files can execute code during loading.


In [ ]:
# Integration uses the counts layer and the precomputed RNA-ATAC graph.
# This tutorial requires cell_type only for labeled plots and evaluation metrics.
# For unlabeled real data, set require_cell_type=False and skip label-based sections.
rna, atac, graph_data, input_summary = load_omics_inputs(
    rna_path=DATA_DIR / "rna_processed.h5ad",
    atac_path=DATA_DIR / "atac_processed.h5ad",
    graph_path=DATA_DIR / "graph_data.pkl",
    require_counts=True,
    require_cell_type=True,
    require_paired=IS_PAIRED,
    paired_task="paired integration",
)

display(input_summary)
print("rna:", rna, "\natac:", atac)


## 3. Prepare the omics and prior graph

scTASL expects raw counts in `X` and stores omics-specific configuration in
`uns["data_config"]`. The dictionary insertion order is intentional: RNA must
precede ATAC so that the feature list matches the prior graph vertex order.


In [ ]:
rna.obs["omics"] = pd.Categorical(["scRNA-seq"] * rna.n_obs)
atac.obs["omics"] = pd.Categorical(["scATAC-seq"] * atac.n_obs)

# Use copies of the count matrices so the original layers remain unchanged.
rna.X = rna.layers["counts"].copy()
atac.X = atac.layers["counts"].copy()

adatas = {"RNA": rna, "ATAC": atac}
for adata_obj in adatas.values():
    configure_dataset(adata_obj, use_obs_names=True)

data_config = {
    omics_name: adata_obj.uns["data_config"]
    for omics_name, adata_obj in adatas.items()
}
vertices = data_config["RNA"]["features"] + data_config["ATAC"]["features"]

graph, train_loader, val_loader = Integration_Model.data_processing(
    adatas=adatas,
    graph_data=graph_data,
)


## 4. Train the integration model

The model automatically uses a CUDA GPU when one is available. Training writes
its log and checkpoints under `RESULT_DIR`. For model hyperparameter tuning,
pass the desired values (for example `latent_dim`, `lr`, or loss weights) to
`Integration_Model` below.


In [ ]:
model = Integration_Model(
    data_config=data_config,
    graph=graph,
    vertices=vertices,
    result_path=str(RESULT_DIR),
    is_paired=IS_PAIRED,
)

model.train_model(
    train_loader=train_loader,
    val_loader=val_loader,
)


## 5. Encode cells and features

Cell embeddings are stored in `obsm["embedding"]`. Feature embeddings are split
according to the configured RNA and ATAC feature counts and stored in
`varm["feature_embedding"]`.


In [ ]:
adatas["RNA"].obsm["embedding"] = model.encode_data(
    model.rna_encoder, adatas["RNA"]
)
adatas["ATAC"].obsm["embedding"] = model.encode_data(
    model.atac_encoder, adatas["ATAC"]
)

feature_embeddings = model.encode_feature()
n_rna_features = len(model.data_config["RNA"]["features"])
n_atac_features = len(model.data_config["ATAC"]["features"])

expected_feature_count = n_rna_features + n_atac_features
if feature_embeddings.shape[0] != expected_feature_count:
    raise ValueError(
        "The number of encoded features does not match the RNA and ATAC "
        "feature configuration."
    )

adatas["RNA"].varm["feature_embedding"] = feature_embeddings[:n_rna_features]
adatas["ATAC"].varm["feature_embedding"] = feature_embeddings[n_rna_features:]

embedding_summary = pd.DataFrame(
    {
        "cell_embedding_shape": [
            adatas["RNA"].obsm["embedding"].shape,
            adatas["ATAC"].obsm["embedding"].shape,
        ],
        "feature_embedding_shape": [
            adatas["RNA"].varm["feature_embedding"].shape,
            adatas["ATAC"].varm["feature_embedding"].shape,
        ],
    },
    index=["RNA", "ATAC"],
)
display(embedding_summary)


### Save omics-specific embeddings

Copies are written so that removing the internal `data_config` metadata does
not alter the in-memory objects needed later in the notebook.


In [ ]:
omics_output_paths = {
    "RNA": ADATA_DIR / "rna_emb.h5ad",
    "ATAC": ADATA_DIR / "atac_emb.h5ad",
}

for omics_name, adata_obj in adatas.items():
    output_adata = adata_obj.copy()
    output_adata.uns.pop("data_config", None)
    output_adata.write(omics_output_paths[omics_name], compression="gzip")
    print(f"Saved {omics_name}: {omics_output_paths[omics_name]}")


## 6. Build and visualize the joint embedding

Each combined observation receives an omics-prefixed index, while its original
identifier is retained in `obs["source_cell_id"]`. This avoids duplicate AnnData
observation names when paired cells share the same barcode.


In [ ]:
combined_obs = []
combined_embeddings = []

for omics_name, adata_obj in adatas.items():
    omics_obs = adata_obj.obs.copy()
    omics_obs["source_cell_id"] = adata_obj.obs_names.astype(str)
    omics_obs.index = pd.Index(
        [f"{omics_name}:{cell_id}" for cell_id in adata_obj.obs_names],
        name="observation_id",
    )
    combined_obs.append(omics_obs)
    combined_embeddings.append(np.asarray(adata_obj.obsm["embedding"]))

combined = anndata.AnnData(
    obs=pd.concat(combined_obs, axis=0, join="inner"),
    obsm={"embedding": np.vstack(combined_embeddings)},
)
combined.obs["omics"] = combined.obs["omics"].astype("category")
combined.obs["cell_type"] = combined.obs["cell_type"].astype("category")

# Build a cosine-neighbor graph and compute UMAP from the learned embedding.
sc.pp.neighbors(combined, use_rep="embedding", metric="cosine")
sc.tl.umap(combined, random_state=RANDOM_SEED)

print(combined)


### Plot omics mixing and cell-type structure

Good integration should mix omics layers while preserving biologically meaningful
cell-type neighborhoods. Figures are displayed inline and saved as editable PDFs.


In [ ]:
combined.uns["omics_colors"] = list(
    sns.color_palette(n_colors=combined.obs["omics"].nunique()).as_hex()
)[::-1]

omics_figure = sc.pl.umap(
    combined,
    color="omics",
    title="Integrated embedding by omics",
    show=False,
    return_fig=True,
)
omics_figure.savefig(
    FIGURE_DIR / "umap_omics.pdf", bbox_inches="tight"
)
display(omics_figure)
plt.close(omics_figure)

cell_type_figure = sc.pl.umap(
    combined,
    color="cell_type",
    palette=cell_type_palette(combined),
    title="Integrated embedding by cell type",
    show=False,
    return_fig=True,
)
cell_type_figure.savefig(
    FIGURE_DIR / "umap_cell_type.pdf", bbox_inches="tight"
)
display(cell_type_figure)
plt.close(cell_type_figure)

combined_path = ADATA_DIR / "combined.h5ad"
combined.write(combined_path, compression="gzip")
print(f"Saved joint embedding: {combined_path}")


## 7. Evaluate integration quality

The evaluation separates two complementary goals:

- **Biology conservation:** cell types remain distinguishable after integration.
- **Omics alignment:** RNA and ATAC omics layers mix without fragmenting cell types.

Higher values indicate better performance for the normalized metrics below.
The final section reports FOSCTTM separately because **lower is better** for that
paired-cell metric.


### Biology conservation

NMI and ARI compare unsupervised clusters with cell-type labels; ASW label,
biological conservation score (BC), and cLISI quantify label separation or
local label purity in the learned embedding.


In [ ]:
embedding = np.asarray(combined.obsm["embedding"])
cell_type_labels = combined.obs["cell_type"]
omics_labels = combined.obs["omics"]

biology_scores = {
    "NMI": normalized_mutual_information(embedding, cell_type_labels),
    "ARI": adjusted_rand_index(embedding, cell_type_labels),
    "ASW label": avg_silhouette_width_label(embedding, cell_type_labels),
    "BC": compute_bcs(embedding, cell_type_labels),
    "cLISI": norm_lisi_label(embedding, cell_type_labels),
}
biology_mean = float(np.mean(list(biology_scores.values())))
display(pd.Series(biology_scores, name="score").to_frame())


### Omics alignment

kBET, graph connectivity (GC), ASW batch, omics entropy (OE), and oLISI measure
omics mixing and neighborhood connectivity. GC uses cell-type labels to check
whether cells of the same biological identity remain connected.


In [ ]:
kbet_score, _, _ = kBET(embedding, omics_labels)
_, omics_entropy_score = compute_omics_entropy(embedding, omics_labels)

omics_alignment_scores = {
    "kBET": kbet_score,
    "GC": graph_connectivity(embedding, cell_type_labels),
    "ASW batch": avg_silhouette_width_batch(
        embedding, omics_labels, cell_type_labels
    ),
    "OE": omics_entropy_score,
    "oLISI": norm_lisi_omics(embedding, omics_labels),
}
omics_alignment_mean = float(np.mean(list(omics_alignment_scores.values())))
overall_score = float(np.mean([biology_mean, omics_alignment_mean]))
display(pd.Series(omics_alignment_scores, name="score").to_frame())


### Label transfer and paired-cell alignment

Transfer accuracy evaluates whether labels learned from one omics layer generalize
across the joint space. FOSCTTM evaluates paired cells and is computed only when
`IS_PAIRED=True`; it relies on the input-order validation performed above.


In [ ]:
transfer_accuracy_score = transfer_accuracy(combined, label_key="cell_type", domain_key="omics")

if IS_PAIRED:
    rna_embedding = np.asarray(adatas["RNA"].obsm["embedding"])
    atac_embedding = np.asarray(adatas["ATAC"].obsm["embedding"])
    foscttm_score = float(foscttm(rna_embedding, atac_embedding))
else:
    foscttm_score = np.nan
    print("FOSCTTM skipped because IS_PAIRED=False.")

print(f"Transfer accuracy: {transfer_accuracy_score:.4f}")
if np.isfinite(foscttm_score):
    print(f"FOSCTTM (lower is better): {foscttm_score:.4f}")


### Summarize and save all metrics

The CSV file is convenient for downstream analysis; the text file is retained
for compatibility with existing result-processing scripts.


In [ ]:
metric_records = [
    *( {"group": "Biology conservation", "metric": name, "score": value}
       for name, value in biology_scores.items() ),
    *( {"group": "Omics alignment", "metric": name, "score": value}
       for name, value in omics_alignment_scores.items() ),
    {
        "group": "Summary",
        "metric": "Biology conservation score",
        "score": biology_mean,
    },
    {
        "group": "Summary",
        "metric": "Omics alignment score",
        "score": omics_alignment_mean,
    },
    {"group": "Summary", "metric": "Overall score", "score": overall_score},
    {
        "group": "Cross-omics evaluation",
        "metric": "Transfer accuracy",
        "score": transfer_accuracy_score,
    },
    {
        "group": "Cross-omics evaluation",
        "metric": "FOSCTTM",
        "score": foscttm_score,
    },
]
metrics_table = pd.DataFrame(metric_records)
display(metrics_table.style.format({"score": "{:.4f}"}))

metrics_csv_path = METRICS_DIR / "integration_metrics.csv"
metrics_txt_path = METRICS_DIR / "integration_metrics.txt"
metrics_table.to_csv(metrics_csv_path, index=False)

with metrics_txt_path.open("w", encoding="utf-8") as handle:
    for row in metrics_table.itertuples(index=False):
        value = "nan" if pd.isna(row.score) else f"{row.score:.4f}"
        handle.write(f"{row.metric}: {value}\n")

print(f"Saved metrics: {metrics_csv_path}")
print(f"Saved compatibility summary: {metrics_txt_path}")


### RNA-to-ATAC label-transfer confusion matrix

A 5-nearest-neighbor classifier is fitted on RNA embeddings and evaluated on
ATAC embeddings. Rows are normalized to percentages, so each row shows how one
true ATAC cell type is distributed across predicted labels.


In [ ]:
confusion_figure_path = FIGURE_DIR / "confusion_matrix.pdf"
confusion_csv_path = FIGURE_DIR / "confusion_matrix.csv"

(
    y_true,
    y_pred,
    confusion_table,
    classification_report_text,
    confusion_accuracy,
) = knn_transfer_confusion_rna_to_atac_percent(
    combined,
    embed_key="embedding",
    label_key="cell_type",
    domain_key="omics",
    rna_name="scRNA-seq",
    atac_name="scATAC-seq",
    k=5,
    metric="cosine",
    weights="uniform",
    l2_normalize=True,
    restrict_to_intersection=True,
    percent=True,
    plot=True,
    save_path=str(confusion_figure_path),
)

print(f"RNA-to-ATAC kNN accuracy: {confusion_accuracy:.4f}")
print(f"Saved confusion matrix: {confusion_csv_path}")



## 8. Next steps

The main reusable outputs are:

- `results/<DATASET>/integration/adata/combined.h5ad` for downstream analysis
- `results/<DATASET>/integration/adata/*_emb.h5ad` for omics-specific analysis
- `results/<DATASET>/integration/metrics/` for quantitative evaluation
- `results/<DATASET>/integration/figure/` for UMAP and label-transfer figures

To process another dataset, first generate the three expected input files, then
update `DATASET` in the configuration cell and rerun the notebook from the top.
